# pyDOE: Design of Experiments for Engineering

## The Core Question

You have a CFD model with 5 input parameters. You have a limited computational budget. **Where should you run your simulations to get the most information?**

Running 10,000 random simulations is wasteful. Running a full factorial is impossible in high dimensions. **Design of Experiments (DOE)** tells you *where* to sample — not just how many times — so that every run gives you maximum information about the parameter space.

## Why pyDOE?

Most engineers reach for random sampling or a naive grid. pyDOE gives you **structured sampling strategies** that are provably better:

- **Full factorial** — every combination, but explodes as 5^n
- **Latin Hypercube (LHS)** — each parameter is uniformly covered, no clustering
- **Sobol sequences** — quasi-random, low-discrepancy, converges faster than random

The key insight: **coverage matters more than count**. 100 well-placed LHS points beat 10,000 random points for most engineering purposes.

## The Workflow

```
Define parameters + bounds
        |
        v
Choose sampling strategy
  (full factorial / LHS / Sobol)
        |
        v
Generate design matrix (N x d)
        |
        v
Scale to physical units
        |
        v
Run simulations at each point
        |
        v
Analyze response surface
```

## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pydoe

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('pyDOE imports OK')

## 1. Full Factorial: The Complete Grid

A **full factorial design** evaluates every combination of parameter levels. For 3 parameters at 3 levels each, that's 3^3 = 27 runs.

**When to use it:** 2-3 parameters, low levels (2-4), and you need to capture all interactions.

**When it breaks:** 5 parameters at 5 levels = 5^5 = **3,125 runs**. 10 parameters at 3 levels = 3^10 = **59,049 runs**. The cost grows exponentially.

The design matrix from `pydoe.fullfact` uses integer level indices (0, 1, 2, ...). You scale them to physical units yourself.


In [ ]:
# 3 parameters, 3 levels each -> 27 runs
levels = [3, 3, 3]
ff = pydoe.fullfact(levels)

print(f'Full factorial: {ff.shape[0]} runs, {ff.shape[1]} parameters')
print('First 6 rows (level indices):')
print(ff[:6].astype(int))

# Scale to physical units
# Param 1: Re = 1e5 to 1e6  (3 levels: 1e5, 5.5e5, 1e6)
# Param 2: alpha = 0 to 20 deg (3 levels: 0, 10, 20)
# Param 3: M = 0.3 to 0.9 (3 levels: 0.3, 0.6, 0.9)
bounds = [[1e5, 1e6], [0, 20], [0.3, 0.9]]
ff_physical = pydoe.scale_samples(ff, bounds)

print('\nFirst 6 rows (physical units):')
print(np.round(ff_physical[:6], 4))

### What does this mean?

The full factorial gives you a **perfect grid** — every corner, every edge, every center is sampled. This is ideal for:
- Building a response surface model (RSM)
- Detecting all 2-way and 3-way interactions
- Validating a surrogate model

But notice: **27 runs for 3 parameters**. Add one more parameter at 3 levels and you're at 81 runs. The exponential growth is the fundamental limitation.


## 2. Latin Hypercube: Uniform Coverage with Fewer Points

A **Latin Hypercube** divides each parameter's range into N equal intervals and places exactly one sample in each interval. This guarantees:

- **Uniform marginal coverage** — no parameter is over- or under-sampled
- **No clustering** — unlike random sampling, points don't clump together
- **N runs for N intervals** — linear scaling, not exponential

For 3 parameters with 20 runs, LHS gives you 20 well-distributed points that cover the entire 3D space.

The `criterion='maximin'` option optimizes the minimum distance between points, giving even better spread.


In [ ]:
# Latin Hypercube: 3 parameters, 20 runs
lhs = pydoe.lhs(3, samples=20, criterion='maximin', iterations=50, seed=42)

print(f'LHS: {lhs.shape[0]} runs, {lhs.shape[1]} parameters')
print('First 5 rows (normalized [0,1]):')
print(np.round(lhs[:5], 4))

# Scale to physical units
lhs_physical = pydoe.scale_samples(lhs, bounds)
print('\nFirst 5 rows (physical units):')
print(np.round(lhs_physical[:5], 4))

### What does this mean?

Compare the LHS to the full factorial:
- **20 runs vs 27 runs** — similar count, but LHS covers the space more evenly
- **No grid structure** — LHS points are scattered, which is better for detecting non-linear effects
- **Linear scaling** — 100 runs for 10 parameters is still just 100 runs

The `maximin` criterion ensures the worst-case (closest) pair of points is as far apart as possible. This is the key property that makes LHS superior to random sampling.


## 3. Sobol Sequence: Quasi-Random, Low Discrepancy

A **Sobol sequence** is a *quasi-random* sequence — it looks random but is actually deterministic and has **low discrepancy**. This means:

- The points fill the space more evenly than random points
- The error of numerical integration converges as O(1/N) instead of O(1/sqrt(N))
- No random seed needed (or use `scramble=True` for randomized Sobol)

Sobol sequences are the gold standard for **global sensitivity analysis** (Sobol indices) and **Monte Carlo integration** in engineering.

> **Note:** `pydoe.sobol_sequence` returns a power-of-2 number of points by default. Use `use_pow_of_2=False` to get exactly N points.


In [ ]:
# Sobol sequence: 3 parameters, 100 runs
sobol = pydoe.sobol_sequence(100, 3, scramble=True, seed=42, use_pow_of_2=False)

print(f'Sobol: {sobol.shape[0]} runs, {sobol.shape[1]} parameters')
print('First 5 rows (normalized [0,1]):')
print(np.round(sobol[:5], 4))

# Scale to physical units
sobol_physical = pydoe.scale_samples(sobol, bounds)
print('\nFirst 5 rows (physical units):')
print(np.round(sobol_physical[:5], 4))

### What does this mean?

Sobol sequences have a **nested structure** — the first 2 points are the corners, the first 4 are the midpoints, the first 8 fill in the next level. This means:

- You can **stop early** if your budget runs out — the first N points are still a good sample
- **Nested designs** — you can refine a study by adding more points without discarding old ones
- **Deterministic** — same seed, same points, every time (reproducibility)

For sensitivity analysis, Sobol sequences are the standard because the Saltelli estimator requires a specific sample structure that Sobol sequences provide naturally.


## 4. Side-by-Side: Coverage Comparison

Let's visualize all three strategies in 2D (first two parameters) to see the difference in coverage.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Full factorial (27 points)
axes[0].scatter(ff_physical[:, 0], ff_physical[:, 1], c='steelblue', s=60, edgecolors='k', linewidths=0.5)
axes[0].set_title(f'Full Factorial\n({ff.shape[0]} runs)', fontsize=11)
axes[0].set_xlabel('Re')
axes[0].set_ylabel('alpha (deg)')

# LHS (20 points)
axes[1].scatter(lhs_physical[:, 0], lhs_physical[:, 1], c='coral', s=60, edgecolors='k', linewidths=0.5)
axes[1].set_title(f'Latin Hypercube\n({lhs.shape[0]} runs)', fontsize=11)
axes[1].set_xlabel('Re')
axes[1].set_ylabel('alpha (deg)')

# Sobol (100 points)
axes[2].scatter(sobol_physical[:, 0], sobol_physical[:, 1], c='seagreen', s=30, edgecolors='k', linewidths=0.3, alpha=0.7)
axes[2].set_title(f'Sobol Sequence\n({sobol.shape[0]} runs)', fontsize=11)
axes[2].set_xlabel('Re')
axes[2].set_ylabel('alpha (deg)')

plt.tight_layout()
plt.show()

### What does this mean?

- **Full factorial** (blue): Perfect grid. Every cell is filled. But only 27 points — in 3D this becomes a sparse lattice.
- **LHS** (coral): 20 points, no grid structure. Each row and column has exactly one point. Good coverage with fewer runs.
- **Sobol** (green): 100 points, quasi-random. Fills the space more evenly than random, with a nested structure visible in the clustering pattern.

**The key trade-off:** Full factorial gives you structure (good for RSM), LHS gives you coverage (good for screening), Sobol gives you both (good for sensitivity analysis).


## 5. Engineering Application: 5-Parameter CFD Study

**The real problem:** You have 5 CFD parameters and a budget of 100 simulation runs. How do you allocate them?

| Parameter | Range | Physical meaning |
|-----------|-------|-----------------|
| Reynolds number | 1e5 - 1e6 | Inertia vs. viscous forces |
| Angle of attack | 0 - 20 deg | Flow incidence |
| Mach number | 0.3 - 0.9 | Compressibility |
| Thickness ratio | 0.05 - 0.20 | Airfoil thickness |
| Surface roughness | 0 - 1e-6 m | Wall roughness |

**Full factorial at 5 levels:** 5^5 = **3,125 runs**. At 4 hours per RANS simulation, that's **520 hours** of compute.

**LHS with 100 runs:** 100 runs = **400 hours**. Same order of magnitude, but 100 well-placed points give you a much better picture of the response surface than 3,125 grid points (which would be sparse in 5D).


In [ ]:
# 5 parameters, 5 levels each
n_params = 5
n_levels = 5
ff_5d = pydoe.fullfact([n_levels] * n_params)
print(f'Full factorial 5D: {ff_5d.shape[0]} runs')

# LHS with 100 runs
lhs_5d = pydoe.lhs(n_params, samples=100, criterion='maximin', iterations=50, seed=42)
print(f'LHS 5D: {lhs_5d.shape[0]} runs')

# Sobol with 100 runs
sobol_5d = pydoe.sobol_sequence(100, n_params, scramble=True, seed=42, use_pow_of_2=False)
print(f'Sobol 5D: {sobol_5d.shape[0]} runs')

# Cost comparison (4 hours per RANS run)
cost_per_run = 4  # hours
print(f'\nCost comparison (4h per run):')
print(f'  Full factorial: {ff_5d.shape[0] * cost_per_run} hours')
print(f'  LHS (100):      {lhs_5d.shape[0] * cost_per_run} hours')
print(f'  Sobol (100):    {sobol_5d.shape[0] * cost_per_run} hours')

### What does this mean?

The full factorial is **31x more expensive** than LHS or Sobol for the same number of parameters. But here's the key insight:

**In 5D, a 5-level full factorial is actually *worse* than 100 LHS points.** Why? Because 3,125 points in 5D is still sparse — the average distance between neighboring points is large. 100 LHS points, spread evenly, give you better coverage of the 5D space.

**Rule of thumb:** For d > 3 parameters, use LHS or Sobol. Full factorial is only practical for d <= 3 with low levels.


## 6. Two-Level Factorial: `ff2n`

For **screening** (which parameters matter?), you often only need two levels: high and low. `pydoe.ff2n(n)` generates a full 2-level factorial for n factors — 2^n runs.

This is the basis of **Plackett-Burman** and **fractional factorial** designs.


In [ ]:
# 4 factors, 2 levels each -> 16 runs
ff2 = pydoe.ff2n(4)
print(f'2-level factorial: {ff2.shape[0]} runs, {ff2.shape[1]} factors')
print('Design matrix (-1 = low, +1 = high):')
print(ff2.astype(int))

# Scale to physical units
bounds_4 = [[1e5, 1e6], [0, 20], [0.3, 0.9], [0.05, 0.20]]
ff2_physical = pydoe.scale_samples(ff2, bounds_4)
print('\nPhysical units (first 4 rows):')
print(np.round(ff2_physical[:4], 4))

### What does this mean?

The 2-level factorial is the **cheapest way to screen** which parameters matter. With 16 runs for 4 factors, you can estimate the main effect of each parameter. If a parameter's effect is small, you can fix it and reduce the problem dimension.

**Workflow:**
1. Run 2-level factorial (16 runs)
2. Rank parameters by effect size
3. Drop parameters with small effects
4. Run LHS or Sobol on remaining parameters (quantitative)


## 7. Exercises

1. **Coverage metric.** Compute the minimum distance between any two points for the full factorial, LHS, and Sobol designs (in 3D, normalized [0,1]). Which design has the largest minimum distance?

2. **Scaling effect.** Generate an LHS with 50 points in 5D. Scale it to the 5 CFD parameters. Plot a 2D projection (Re vs alpha) and check: does each parameter's marginal distribution look uniform?

3. **Nested Sobol.** Generate a Sobol sequence with 128 points. Plot the first 16, first 32, and first 128 points. Verify the nested structure — are the first 16 points a subset of the first 32?

4. **Fractional factorial.** Use `pydoe.fracfact` to generate a fractional factorial design for 5 factors. How many runs does it use? What resolution does it have?

5. **Real CFD.** Replace the surrogate model with a call to an actual CFD solver (or a pre-computed dataset). Generate an LHS with 50 points, run the simulations, and fit a polynomial response surface. How well does it predict?
